Phase 4 Don't Need the GPU , Implementation will be continued using the local CPU

In [4]:
# ==========================================
# 1. IMPORTS
# ==========================================

import os
import sys
import json
import random
import numpy as np
import pandas as pd
import torch
import kaggle
from datasets import Dataset

from pathlib import Path
from dotenv import load_dotenv

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# ==========================================
# 2. PROJECT CONFIGURATION
# ==========================================

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ==========================================
# 3. ENVIRONMENT VARIABLES
# ==========================================

load_dotenv(PROJECT_ROOT / ".env")

HF_TOKEN = os.getenv("HF_TOKEN")
KAGGLE_API_TOKEN = os.getenv("KAGGLE_API_TOKEN")

# ==========================================
# 4. RANDOM SEED
# ==========================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Project initialized successfully!")

Project initialized successfully!


Configure model and dataset

In [5]:
MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

DATA_DIR = PROJECT_ROOT / "data" / "prepraed"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

Download Dataset from Kaggle

In [6]:

# Download dataset from Kaggle

DATA_DIR = PROJECT_ROOT / "data" / "prepared"
DATA_DIR.mkdir(parents=True, exist_ok=True)

KAGGLE_DATASET = "dgowda11/sms-spam-cleaned"

!kaggle datasets download -d {KAGGLE_DATASET} -p "{DATA_DIR}" --unzip

print("Dataset downloaded to:", DATA_DIR)
print("Files:", list(DATA_DIR.iterdir()))


^C
Dataset downloaded to: c:\Users\INDAML\AI\llama-spam-finetuning\data\prepared
Files: [WindowsPath('c:/Users/INDAML/AI/llama-spam-finetuning/data/prepared/test.csv'), WindowsPath('c:/Users/INDAML/AI/llama-spam-finetuning/data/prepared/train.csv')]


Load DataSet

In [7]:
train_df  = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

print("Train DataFrame shape:", train_df.shape)
print("Test DataFrame shape:", test_df.shape)

train_df.head()

Train DataFrame shape: (4135, 2)
Test DataFrame shape: (1034, 2)


,label,message
0,ham,Cool. So how come you havent been wined and di...
1,ham,Huh so late... Fr dinner?
2,ham,my ex-wife was not able to have kids. Do you w...
3,ham,Hope ur head doesn't hurt 2 much ! Am ploughin...
4,ham,Ryder unsold.now gibbs.


Create the Prompt Formatter 

In [8]:
# Load the Tokenizer from Hugging Face

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN
)

print("Tokenizer loaded:", tokenizer.__class__.__name__)
print("System prompt:", tokenizer.vocab_size)
print("EOS Token:", tokenizer.eos_token)

Tokenizer loaded: TokenizersBackend
System prompt: 128000
EOS Token: <|eot_id|>


In [18]:
# Instruction formatting configuration 

SYSTEM_PROMPT = (
    "You are an SMS Classification assistant."
    "Classify the message as HAM or SPAM."
    "Respond only with HAM or SPAM."
)

def format_chat_template(message, label):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"Message: {message}\nLabel: {label}"
        },
        {
            "role": "assistant",
            "content": label.upper()
        }
    ]
    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return formatted_text

                                                   

In [19]:
train_df["formatted_text"] = train_df.apply(
    lambda row: format_chat_template(row["message"], row["label"]),
    axis=1,
)

In [14]:

# Inspect one HAM and one SPAM sample

ham_sample = train_df[train_df["label"] == "ham"].iloc[0]
spam_sample = train_df[train_df["label"] == "spam"].iloc[0]

print("HAM SAMPLE:\n")
print(format_chat_template(ham_sample["message"], ham_sample["label"]))

print("\n" + "=" * 80 + "\n")

print("SPAM SAMPLE:\n")
print(format_chat_template(spam_sample["message"], spam_sample["label"]))


HAM SAMPLE:

{'input_ids': [128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220, 1114, 4723, 220, 2366, 21, 271, 2675, 527, 459, 31009, 55982, 18328, 19891, 1463, 279, 1984, 439, 473, 1428, 477, 9440, 1428, 8542, 3595, 1193, 449, 473, 1428, 477, 9440, 1428, 13, 128009, 128006, 882, 128007, 271, 2097, 25, 24882, 13, 2100, 1268, 2586, 499, 31081, 306, 1027, 289, 1619, 323, 294, 1619, 1603, 5380, 2535, 25, 13824, 128009, 128006, 78191, 128007, 271, 74593, 128009], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


SPAM SAMPLE:

{'input_ids': [128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220, 1114, 4723, 220, 2366, 21, 271, 2675, 527, 459, 31009, 559

Convert Pandas dataset into the Hugging face format dataset 

In [20]:
# Convert pandas Dataframe into the Hugging Face Dataset format
train_dataset = Dataset.from_pandas(
    train_df[["label","message","formatted_text"]],
    preserve_index=False,
)

test_dataset = Dataset.from_pandas(
    test_df[["label","message"]],
    preserve_index=False,
)

print("Traning Dataset:", train_dataset)
print("Testing Dataset:", test_dataset)


Traning Dataset: Dataset({
    features: ['label', 'message', 'formatted_text'],
    num_rows: 4135
})
Testing Dataset: Dataset({
    features: ['label', 'message'],
    num_rows: 1034
})


In [26]:
print("Column names ",train_dataset.column_names)
print("Number of rows ", len(train_dataset))

print(train_dataset[0])

Column names  ['label', 'message', 'formatted_text']
Number of rows  4135
{'label': 'ham', 'message': 'Cool. So how come you havent been wined and dined before?', 'formatted_text': '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 17 Nov 2026\n\nYou are an SMS Classification assistant.Classify the message as HAM or SPAM.Respond only with HAM or SPAM.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nMessage: Cool. So how come you havent been wined and dined before?\nLabel: ham<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nHAM<|eot_id|>'}


Tokenization of Dataset

In [28]:
MAX_SEQUENCE_LENGTH = 512

In [30]:
# Define the maximum sequence length 
def tokenize_training_example(example):
    tokenized = tokenizer(
        example["formatted_text"],
        truncation=True,
        max_length=MAX_SEQUENCE_LENGTH,
    )
    return tokenized

train_tokenized = train_dataset.map(
    tokenize_training_example,
    batched = True,
    desc = "Tokenizing training dataset"
)

print(train_tokenized)

Tokenizing training dataset: 100%|██████████| 4135/4135 [00:02<00:00, 1389.24 examples/s]

Dataset({
    features: ['label', 'message', 'formatted_text', 'input_ids', 'attention_mask'],
    num_rows: 4135
})


Following is the Inference Prompt containing only the System and User message   

In [33]:
def tokenize_training_example(examples):
    prompts= []

    for message in examples["message"]:

        messages = [
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": f"message:{message}",
            }
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        prompts.append(prompt)

    tokenized = tokenizer(
        prompts,
        truncation=True,
        padding=False,
        add_special_tokens=False,
    )

    return tokenized

test_tokenized = test_dataset.map(
    tokenize_training_example,
    batched = True,
    desc = "Tokenizing test dataset"
)
print(test_tokenized)

Tokenizing test dataset: 100%|██████████| 1034/1034 [00:00<00:00, 7571.38 examples/s]

Dataset({
    features: ['label', 'message', 'input_ids', 'attention_mask'],
    num_rows: 1034
})


In [34]:
sample = train_tokenized[0]

print("Input IDs:")
print(sample["input_ids"])

print("\nAttention Mask:")
print(sample["attention_mask"])

print("\nNumber of tokens:", len(sample["input_ids"]))

Input IDs:
[128000, 128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220, 1114, 4723, 220, 2366, 21, 271, 2675, 527, 459, 31009, 55982, 18328, 19891, 1463, 279, 1984, 439, 473, 1428, 477, 9440, 1428, 8542, 3595, 1193, 449, 473, 1428, 477, 9440, 1428, 13, 128009, 128006, 882, 128007, 271, 2097, 25, 24882, 13, 2100, 1268, 2586, 499, 31081, 306, 1027, 289, 1619, 323, 294, 1619, 1603, 5380, 2535, 25, 13824, 128009, 128006, 78191, 128007, 271, 74593, 128009]

Attention Mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

Number of tokens: 85


In [ ]:
# Decode the token IDs 
decode_text = tokenizer.decode(sample["input_ids"], skip_special_tokens=True)
print(decode_text)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


system

Cutting Knowledge Date: December 2023
Today Date: 17 Nov 2026

You are an SMS Classification assistant.Classify the message as HAM or SPAM.Respond only with HAM or SPAM.user

Message: Cool. So how come you havent been wined and dined before?
Label: hamassistant

HAM


In [37]:
# validate the tokenization
assert len(train_tokenized) == 4135
assert len(test_tokenized) == 1034

# Required columns
assert "input_ids" in train_tokenized.column_names
assert "attention_mask" in train_tokenized.column_names

assert "input_ids" in test_tokenized.column_names
assert "attention_mask" in test_tokenized.column_names

# Sequence length validation
train_lengths = [
    len(ids) for ids in train_tokenized["input_ids"]
]

test_lengths = [
    len(ids) for ids in test_tokenized["input_ids"]
]

assert max(train_lengths) <= MAX_SEQUENCE_LENGTH
assert max(test_lengths) <= MAX_SEQUENCE_LENGTH

# Attention mask validation
assert all(
    len(ids) == len(mask)
    for ids, mask in zip(
        train_tokenized["input_ids"],
        train_tokenized["attention_mask"]
    )
)

print("Tokenization validation successful!")

Tokenization validation successful!


Analyze sequence lengths

In [38]:
print("Training Token Length Statistics:")
print(pd.Series(train_lengths).describe())

print("\nTesting Token Length Statistics:")
print(pd.Series(test_lengths).describe())

Training Token Length Statistics:
count    4135.000000
mean       91.495768
std        16.757654
min        71.000000
25%        79.000000
50%        86.000000
75%       101.000000
max       328.000000
dtype: float64

Testing Token Length Statistics:
count    1034.000000
mean       85.535783
std        17.786710
min        65.000000
25%        73.000000
50%        80.000000
75%        95.000000
max       269.000000
dtype: float64
